In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import seaborn as sns

In [ ]:
#Cargamos los datos
df=pd.read_csv("world_bank_country_data.csv")

In [ ]:
#Hacemos un analisis descriptivo
df.head()

In [ ]:
df.info()

In [ ]:
#Miramos si hay valores nulos
df.isna().sum()

In [ ]:
#La deteccion de outliers esta en la correlacion

In [ ]:
#Filtramos el pais a solo paises bajos
df= df[df["country"] == "Netherlands"]


In [ ]:
#En el archivo de correlacion, nos dio que los que mas correlacion tienen con la variable son los que vamos a elegir para la regresion:
indicadores_elegidos = ["Commercial service exports (current US$)","High-technology exports (% of manufactured exports)","Merchandise exports (current US$)","ICT goods exports (% of total goods exports)","Merchandise imports (current US$)","Population growth (annual %)","External balance on goods and services (current US$)"]
df = df[df["indicator"].isin(indicadores_elegidos)]
df

In [ ]:
#cargamos los datos del puerto y cambiamos la fechas a datetime para un analisis correcto
puerto = pd.read_csv("datosPuerto.csv", decimal=",")
puerto['FECHA'] = pd.to_datetime(puerto['FECHA'])
puerto['FECHA'] = puerto['FECHA'].dt.year

In [ ]:
#Filtramos solo importaciones y agrupamos los kilos por cada año 
importaciones = puerto[(puerto["PAIS_FINAL"] == "ES") & (puerto["PAIS_ORIGEN"] == "NL")]
importaciones = importaciones.groupby("FECHA")["PESO_KILOS"].sum()
importaciones

In [ ]:
#Lo pasamos a un diccionario
kilos_importaciones = {  
2009: 826407028,  
2010: 1096816180,  
2011: 786733730,  
2012: 925922346,  
2013: 732426144,  
2014: 785162500,  
2015: 790342418,  
2016: 690588712,  
2017: 683978612,  
2018: 696356457,  
2019: 831587812,  
2020: 520076382,  
2021: 655221933,  
2022: 254646377,  
2023: 323879014}

In [ ]:
#Ponemos los kilos para que cuadren con el dataset de indicadores
df_kilos = pd.Series(kilos_importaciones).reset_index()
df_kilos.columns = ["date", "kilos"]

In [ ]:
#hacemos un pivot_table y un merge para que se junten los datos de los kilos y los indicadores
df_pivot = df.pivot_table(index="date",columns="indicator",values="value").reset_index()
df_merge = df_kilos.merge(df_pivot, on="date", how="left")
df_merge

In [ ]:
#Imputamos los Nan con la media
df_merge["High-technology exports (% of manufactured exports)"] = df_merge["High-technology exports (% of manufactured exports)"].fillna(
    df_merge["High-technology exports (% of manufactured exports)"].mean())

df_merge["ICT goods exports (% of total goods exports)"] = df_merge["ICT goods exports (% of total goods exports)"].fillna(
    df_merge["ICT goods exports (% of total goods exports)"].mean())

In [ ]:
df_merge.corr()
#vamos a elegir los que mas correlacio n tienen con los kilos, y vamos a ignorar los que sean parecidos como Merchandise exports (current US$ y Merchandise imports (current US$) y nos vamos a quedar solo con uno

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import RobustScaler

In [ ]:
#Separamos en x e y(en x ponemos los 4 indicadores que hemos elegido por su correlacionr)
X = df_merge[["Commercial service exports (current US$)","External balance on goods and services (current US$)","High-technology exports (% of manufactured exports)","Merchandise imports (current US$)","Population growth (annual %)"]]
y = df_merge["kilos"]

In [ ]:
#separamos en train y test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [ ]:
# Escalamos los datos con el RobustScaler
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
#Creamos y entrenamos el modelo
modelo = LinearRegression()
modelo.fit(X_train_scaled, y_train)

In [ ]:
#predecimos el modelo
y_pred_lr = modelo.predict(X_test_scaled)

In [ ]:
#visualizamos las metricas importantes
print("MAE", mean_absolute_error(y_test, y_pred_lr))
print("MSE", mean_squared_error(y_test, y_pred_lr))
print("R2", r2_score(y_test, y_pred_lr))

In [ ]:
#Calculamos el intercepto para sacar la formula de regresion lineal
print("Intercept:", modelo.intercept_)


In [ ]:
#sacamos el coeficiente de cada variable para ver cuanto influyen los indicadores en los kilos
nombres = X_train.columns
valores = modelo.coef_
for i in range(len(nombres)):
    print(nombres[i], valores[i])

In [ ]:
#Hacemos el grafico para visualizar la importacia de cada variable con los kilos
plt.figure(figsize=(12,6))
plt.bar(nombres, valores, color='orange')
plt.xticks(rotation=45)
plt.ylabel("coeficiente")
plt.title("Coeficientes por variable(Importaciones)")
plt.tight_layout()
plt.show()